## BERT Fine-Tuning

This notebook showcases BERT fine-tuning with a grid-search hyperparameter sweep. This setup is identical to the notebook used for fine-tuning FinBERT on the PhaseBank dataset in order to make a fair comparison between performances.

The grid-search finds the optimal hyperparameters and assesses model performance on the validation set to determine which configurations should be used for the final predictions on the test set. 

The learning rates also spans from the set `{1e-5, 2e-5, 5e-5}` and `3 to 5 epochs.`

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('financial-sentiment-comparison'):
        !git clone https://github.com/maximusrome/financial-sentiment-comparison.git
    %cd financial-sentiment-comparison
    !pip install -q transformers==4.44.2 pandas scikit-learn pyarrow matplotlib seaborn

from pathlib import Path
repo_root = Path.cwd()
if repo_root.name == "notebooks":
  repo_root = repo_root.parent
if str(repo_root) not in sys.path:
  sys.path.insert(0, str(repo_root))
print("cwd:", os.getcwd())
print("repo_root:", repo_root)
print("sys.path[0:5]:", sys.path[:5])
from evaluation import evaluate_predictions
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

## Load Data

In [ ]:
from data_loader import build_and_save_split, load_split, SPLIT_PATH
if not SPLIT_PATH.exists():
    build_and_save_split()

## Run Sweep

In [ ]:
from finbert_train import run_sweep

sweep_df = run_sweep(
    model_name='google-bert/bert-base-uncased',
    output_name='bert_finetuned',
    learning_rates=[1e-5, 2e-5, 5e-5],
    num_epochs_list=[3, 4],
    batch_size=16,
    seed=42,
    sweep_results_path="results/tables/bert_sweep.csv"
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Visualizations

In [ ]:
import pandas as pd
sweep_df = pd.read_csv('results/tables/bert_sweep.csv')
sweep_df.round(4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 4))
for n_ep, group in sweep_df.groupby('num_epochs'):
    g = group.sort_values('learning_rate')
    ax.plot(g['learning_rate'], g['val_f1'], marker='o',
            label=f'{n_ep} epochs')
ax.set_xscale('log')
ax.set_xlabel('Learning rate (log scale)')
ax.set_ylabel('Validation macro-F1')
ax.set_title('BERT hyperparameter sweep')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/bert_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Accuracy Results + F1 Score

In [ ]:
from evaluation import evaluate_predictions
result = evaluate_predictions(f'{repo_root}/predictions/bert_finetuned_predictions.csv')
print(f"Test accuracy:  {result['overall']['accuracy']:.4f}")
print(f"Test macro-F1:  {result['overall']['macro_f1']:.4f}\n")
print('By agreement tier:')
for tier, m in result['by_tier'].items():
    print(f"  tier={tier:6s}  acc={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}  (n={m['n']})")

In [ ]:
!zip -r /content/output.zip data predictions results
from google.colab import files
files.download('/content/output.zip')